In [ ]:
#!unzip -q "xxxxxxxxxxxxxxxxx" -d "/xxxxxxxxxxxxx/" # Descomprimir el archivo de datos

In [ ]:
# Leer datasets
from torchvision import datasets, transforms

# Transformaciones: convertir cada imagen a Tensor de PyTorch
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Leer set de entrenamiento
PATH_TRAIN = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
train_dataset = datasets.ImageFolder(root=PATH_TRAIN,
                                     transform=transform)

# Leer set de prueba
PATH_TEST = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
test_dataset = datasets.ImageFolder(root=PATH_TEST,
                                    transform=transform)

#Mostrar en pantalla el tamaño del dataset
len(train_dataset), len(test_dataset)

In [ ]:
print(test_dataset.class_to_idx)

In [ ]:
import random
import matplotlib.pyplot as plt

idx_to_class = {v: k for k, v in test_dataset.class_to_idx.items()}

idxs = random.sample(range(len(test_dataset)), 6)

# Mostrar las imagenes
fig, ax = plt.subplots(2, 3, figsize=(12,8))
for i in range(6):
    img, lbl = test_dataset[idxs[i]]
    ax[i//3, i%3].imshow(img.permute(1, 2, 0))
    ax[i//3, i%3].set_title(f"etiqueta: {idx_to_class[lbl]}")
    ax[i//3, i%3].axis("off")
plt.show()

In [ ]:
print(img.shape)
print(img.min(), img.max())

In [ ]:
import torch
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=4),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=4),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(256, 256, kernel_size=3),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=5, stride=2, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 3, kernel_size=5, stride=2, output_padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [ ]:
from torchsummary import summary

model = Autoencoder().cuda() # Quitamos .cuda() para que corra en CPU
summary(model, input_size=(3, 224, 224)) # Especificamos device="cpu"

In [ ]:
from torch.utils.data import DataLoader

# crear dataloader del set de entrenamiento
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

#Crear instancia del modelo y moverlo a la GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = Autoencoder().to(device)

# Definir pérdida y optimizador
perdida = nn.MSELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

# Iteraciones de entrenamiento
n_its = 100
for epoch in range(n_its):
    modelo.train()
    acumulador_perdida = 0.0
    for imgs, _ in train_loader:
        # Mover lote de imagenes a la GPU
        imgs = imgs.to(device)

        # Generar predicciones y calcular perdida
        preds = modelo(imgs)
        loss = perdida(preds, imgs)

        # Actualizar parametros del modelo
        optimizador.zero_grad()
        loss.backward()
        optimizador.step()

        # Y almacenar pérdida en el acumulador
        acumulador_perdida += loss.item() * imgs.size(0)

    # Terminada la iteración de entrenamineto, calcular la perdida de esta iteración
    epoch_loss = acumulador_perdida / len(train_loader.dataset)

    # Imprimir progreso en la pantalla
    print(f"Epoch [{epoch+1}/{n_its}], Perdida: {epoch_loss:.6f}")

In [ ]:
# Recordar las categorias asignadas a normales y anomalas
print(test_dataset.class_to_idx)

# Ejemplo de predicción sobre un dato normaly sobre uno anormal
img_norm, lbl_norm = test_dataset[21]
img_anomala, lbl_anom = test_dataset[0]

print(lbl_norm)
print(lbl_anom)

In [ ]:
# Mover imágenes a la GPU
img_norm = img_norm.unsqueeze(0).to(device)  # 3 x 224 x 224
img_anomala = img_anomala.unsqueeze(0).to(device)

# Reconstruir imágenes
recon_norm = modelo(img_norm)
recon_anomala = modelo(img_anomala)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

# Normal
ax[0, 0].imshow(img_norm.squeeze().permute(1, 2, 0).cpu().numpy())
ax[0, 1].imshow(recon_norm.squeeze().permute(1, 2, 0).detach().cpu().numpy())
ax[0, 0].set_title("Normal - Original")
ax[0, 1].set_title("Normal - Reconstruida")

# Anómala
ax[1, 0].imshow(img_anomala.squeeze().permute(1, 2, 0).cpu().numpy())
ax[1, 1].imshow(recon_anomala.squeeze().permute(1, 2, 0).detach().cpu().numpy())
ax[1, 0].set_title("Anómala - Original")
ax[1, 1].set_title("Anómala - Reconstruida")

# Quitar ejes
for i in range(2):
    for j in range(2):
        ax[i, j].axis('off')

plt.tight_layout();

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

# Normal
ax[0, 0].imshow(img_norm.squeeze()[:,:-10,:-10].permute(1, 2, 0).cpu().numpy())
ax[0, 1].imshow(recon_norm.squeeze()[:,:-10,:-10].permute(1, 2, 0).detach().cpu().numpy())
ax[0, 0].set_title("Normal - Original")
ax[0, 1].set_title("Normal - Reconstruida")

# Anómala
ax[1, 0].imshow(img_anomala.squeeze()[:,:-10,:-10].permute(1, 2, 0).cpu().numpy())
ax[1, 1].imshow(recon_anomala.squeeze()[:,:-10,:-10].permute(1, 2, 0).detach().cpu().numpy())
ax[1, 0].set_title("Anómala - Original")
ax[1, 1].set_title("Anómala - Reconstruida")

# Quitar ejes
for i in range(2):
    for j in range(2):
        ax[i, j].axis('off')

plt.tight_layout();

In [ ]:
import torch.nn.functional as F
from tqdm import tqdm

# Crear dataloader para el set de prueba
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# LLevar el modelo al modo "inferencia"
modelo.eval()

# Generar predicciones
categorias = []
errores = []
with torch.no_grad():
  for img, label in tqdm(test_loader):
    img = img.to(device)
    pred = modelo(img)

    # Error cuadraticos medio sin artefactos
    loss = F.mse_loss(pred[:,:,:-10,:-10], img[:,:,:-10,:-10], reduction='mean').item()

    #Almacenar perdida y categoria de la imagen
    errores.append(loss)
    categorias.append(label.item())


In [ ]:
import numpy as np

errores = np.array(errores)
categorias = np.array(categorias)

plt.hist(errores[categorias==0], bins=50, alpha=0.5, label='bad')
plt.hist(errores[categorias==1], bins=50, alpha=0.5, label='good')
plt.legend()
plt.title("Distribución de errores")
plt.xlabel("Error cuadratico medio")
plt.ylabel("Conteo");

In [ ]:
def calcular_mapa_anomalias(img, rec):
  #calcular diferencias cuadraticas
  dif_cuad = (img - rec)**2

  # Reducir de 3 a 1 canal tomando el max
  mapa = dif_cuad.max(dim=1)[0]

  return mapa

In [ ]:
# Imagen normal y anomala
img_norm, lbl_norm = test_dataset[45]
img_anomala, lbl_anom = test_dataset[10]

# Reconstrucciones
img_norm = img_norm.unsqueeze(0).to(device)
img_anomala = img_anomala.unsqueeze(0).to(device)

recon_norm = modelo(img_norm)
recon_anomala = modelo(img_anomala)

# Mapa de anomalias (eliminar artefactis)
mapa_normal = calcular_mapa_anomalias(img_norm, recon_norm)[:,:-10,:-10]
mapa_anormal = calcular_mapa_anomalias(img_anomala, recon_anomala)[:,:-10,:-10]

# Calcular promedio de cada mapa de anomalias
mapa_normal_mean = mapa_normal.mean()
mapa_anormal_mean = mapa_anormal.mean()

# Graficar mapas de anomalias
fig, ax = plt.subplots(2, 2, figsize=(12, 8))


# Normal
ax[0, 0].imshow(img_norm[0].permute(1, 2, 0).cpu().numpy())
ax[0, 1].imshow(mapa_normal[0].cpu().detach().numpy(), cmap="jet", vmax=mapa_normal.max().item())
ax[0, 0].set_title("Normal - Original")
ax[0, 1].set_title(f"Normal - Mapa de anomalias - Promedio ={mapa_normal_mean:.6f}")

# Anómala
ax[1, 0].imshow(img_anomala[0].permute(1, 2, 0).cpu().numpy())
ax[1, 1].imshow(mapa_anormal[0].cpu().detach().numpy(), cmap="jet", vmax=mapa_normal.max().item())
ax[1, 0].set_title("Anómala - Original")
ax[1, 1].set_title(f"Anomala - Mapa de anomalias - Promedio ={mapa_anormal_mean:.6f}")

# Quitar ejes
for i in range(2):
    for j in range(2):
        ax[i, j].axis('off')

plt.tight_layout();


In [ ]:
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Llevar el modelo al modo "inferencia"
modelo.eval()

# Generar predicciones
categorias = []
errores = []
with torch.no_grad():
  for img, label in tqdm(test_loader):
    img = img.to(device)
    pred = modelo(img)

    # Error cuadraticos medio sin artefactos
    mapa_anomala = calcular_mapa_anomalias(img, pred)[:,:-10,:-10]
    loss = mapa_anomala.mean().item()

    #Almacenar perdida y categoria de la imagen
    errores.append(loss)
    categorias.append(label.item())

# Graficas distribuciones
errores = np.array(errores)
categorias = np.array(categorias)

plt.hist(errores[categorias==0], bins=50, alpha=0.5, label='bad')
plt.hist(errores[categorias==1], bins=50, alpha=0.5, label='good')
plt.legend()
plt.title("Distribución de errores de reconstruccion (mapa de anomalias)")
plt.xlabel("Promedio mapa de anomalias")
plt.ylabel("Conteo");